# Option C - GRPO with a verifiable reward (fast check)

This is the "different finetuning approach" from the notebook you downloaded (`Qwen3_5_(4B)_Vision_GRPO.ipynb`). Instead of SFT on ~20 noisy labels, it optimises a **reward** you can compute *without* per-sample ground truth - which is precisely the constraint you're under.

**The RL training loop is deferred** (it's the highest-effort, least-certain path, and it needs a decent base extractor + an eval set first). The thing worth checking *now* is cheap and mostly runs on CPU: **do the reward functions actually carry signal?** If a broken table doesn't score lower than a good one, GRPO would just optimise noise.

## The three rewards, and why none needs a label
- **structure validity** - parses, is table-shaped, cells tile the grid. Pure structure.
- **totals reconciliation** - `sum(line items) == total`. An invoice carries its own checksum.
- **TEDS-Struct vs a reference** - structural agreement with the teacher draft (or a second model). Rewards consensus, not a human label.

In [ ]:
import sys, re
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.data.html_utils import extract_cells, parse_table
from src.model.prompts import clean_prediction
from src.eval.teds import teds_score


def reward_structure_validity(html: str) -> float:
    """1.0 = parses, is table-shaped (>=2x2), and its cells tile the grid without
    big holes. Partial credit so GRPO sees a gradient, not a cliff. Needs no label."""
    if parse_table(html) is None:
        return 0.0
    cells = extract_cells(html)
    if not cells:
        return 0.0
    n_rows = max(c.row + c.rowspan for c in cells)
    n_cols = max(c.col + c.colspan for c in cells)
    score = 0.5                                   # parsed at all
    if n_rows >= 2 and n_cols >= 2:
        score += 0.25                             # table-shaped
    covered = sum(c.rowspan * c.colspan for c in cells)
    fill = covered / (n_rows * n_cols) if (n_rows and n_cols) else 0.0
    score += 0.25 * min(1.0, fill)                # cells tile the grid
    return round(score, 3)


_NUM = re.compile(r'-?\d[\d,]*\.?\d*')

def _numbers(text):
    out = []
    for m in _NUM.findall(text):
        try:
            out.append(float(m.replace(',', '')))
        except ValueError:
            pass
    return out


def reward_totals_reconcile(html: str):
    """Invoice self-checksum: does the amount on a 'total' row equal the sum of the
    numbers in its column above it? Reward in [0,1], or None when no total is found
    (the signal is only defined for invoices that carry it). No label needed.

    This is a *template sniff test* -- validate that it separates good from broken
    on YOUR invoice layouts before trusting it as a GRPO reward; real invoices with
    subtotal+tax+total rows need the column logic adapted."""
    cells = extract_cells(html)
    if not cells:
        return None
    total_row = None
    for c in cells:
        if re.search(r'\b(total|amount due|balance|grand total)\b', c.text, re.I):
            total_row = c.row                     # last matching row wins
    if total_row is None:
        return None
    total_val = total_col = None
    for c in sorted((x for x in cells if x.row == total_row), key=lambda x: x.col):
        ns = _numbers(c.text)
        if ns:
            total_val, total_col = ns[-1], c.col  # rightmost number on the total row
    if total_val is None:
        return None
    col_vals = []
    for c in cells:
        if c.row < total_row and c.col <= total_col < c.col + c.colspan:
            ns = _numbers(c.text)
            if ns:
                col_vals.append(ns[-1])
    if not col_vals:
        return None
    err = abs(sum(col_vals) - total_val) / max(abs(total_val), 1.0)
    return round(max(0.0, 1.0 - err), 3)


def reward_teds_vs_reference(html: str, reference_html: str) -> float:
    """Structural agreement with a reference draft (the teacher, or a second model).
    Rewards consensus on structure -- no human label required."""
    try:
        return round(teds_score(html, reference_html, structure_only=True), 3)
    except ValueError:
        return 0.0


## The fast check: do the rewards *discriminate*?
A reward is only useful if a worse table scores lower. A clean reconstruction and a deliberately broken one (header collapsed, a line item dropped) go through each reward - the good one must win.

In [ ]:
# A clean reconstruction and a broken one (header collapsed, a line item dropped).
GOOD = (
    '<table>'
    '<tr><th>Item</th><th>Qty</th><th>Price</th><th>Amount</th></tr>'
    '<tr><td>Widget</td><td>2</td><td>10.00</td><td>20.00</td></tr>'
    '<tr><td>Gadget</td><td>1</td><td>30.00</td><td>30.00</td></tr>'
    '<tr><td colspan="3">Total</td><td>50.00</td></tr>'
    '</table>'
)
BROKEN = (
    '<table>'
    '<tr><td>Item Qty Price Amount</td></tr>'
    '<tr><td>Widget 2 10.00 20.00</td></tr>'
    '<tr><td>Total 50.00</td></tr>'          # Gadget line lost -> 20 != 50
    '</table>'
)

print(f'{"":8s} validity  teds_vs_good  totals')
for tag, html in [('good', GOOD), ('broken', BROKEN)]:
    print(f'{tag:8s} {reward_structure_validity(html):8}  '
          f'{reward_teds_vs_reference(html, GOOD):12}  {reward_totals_reconcile(html)}')

# Expect: good scores higher on every defined reward. If a reward does NOT separate
# them, it cannot train GRPO -- drop it or fix it before the RL loop.


## Score your real labels (optional)
If you produced `data/teacher-grounded/labels.jsonl` in Option A, this scores each label - a cheap, unlabelled quality gate that doubles as the GRPO reward.

In [ ]:
import json

MANIFEST = ROOT / 'data' / 'teacher-grounded' / 'labels.jsonl'   # from Option A
if MANIFEST.exists():
    recs = [json.loads(l) for l in MANIFEST.read_text().splitlines() if l.strip()]
    print(f'{len(recs)} labels\n')
    print(f'{"uid":24s} validity totals')
    for r in recs:
        h = r['html']
        print(f'{r["uid"][:24]:24s} {reward_structure_validity(h):8} {reward_totals_reconcile(h)}')
    print('\nLow validity or a failed totals check flags a label to fix by hand '
          '(and is exactly the signal GRPO would optimise).')
else:
    print('No manifest yet -- run the Option A batch first. '
          'Rewards are already validated on the synthetic pair above.')


## Wiring GRPO (deferred - for reference)
Once the rewards discriminate on your data, they plug into `GRPOTrainer` as `reward_funcs`. Left commented; run only after the gates below.

In [ ]:
# GPU box only, and ONLY after the two gates below are met. This mirrors the
# downloaded Qwen3_5_(4B)_Vision_GRPO.ipynb -- the "different finetuning approach".
# It optimises a REWARD, so it needs no per-sample ground-truth label; that is
# exactly why it fits your no-labels situation. Left commented on purpose.
#
# from trl import GRPOConfig, GRPOTrainer
# from unsloth import FastVisionModel
#
# def reward_fn(completions, reference_htmls=None, **kw):
#     htmls = [clean_prediction(c) for c in completions]
#     rewards = []
#     for h in htmls:
#         v = reward_structure_validity(h)
#         t = reward_totals_reconcile(h)
#         rewards.append(0.6 * v + 0.4 * (t if t is not None else 0.0))
#     return rewards
#
# model, processor = FastVisionModel.from_pretrained('Qwen/Qwen3-VL-8B-Instruct',
#                                                     load_in_4bit=True)
# trainer = GRPOTrainer(model=model, reward_funcs=[reward_fn],
#                       args=GRPOConfig(max_prompt_length=4096, num_generations=4, ...),
#                       train_dataset=...)   # prompts only -- no target HTML needed
# trainer.train()


---
**Gates before running GRPO for real:**
1. The rewards above **discriminate on your invoices**, not just the synthetic pair.
2. The base extractor (from Option A / B) is already **decent** - rewards sit near zero on garbage, so RL has nothing to climb otherwise.
3. The **20-invoice eval set** exists, so you can tell a real gain from reward hacking.

Until all three hold, Option A (grounding) and Option B (family swap) are the higher-certainty moves. GRPO is the last lever, not the first.